# 05 — Particle & Field: strain-path tracking

The `Sequence` gives you displacements at **fixed mesh nodes** for each image pair.
`Particle` and `Field` go further: they integrate those displacements along
**material point trajectories** to build up the full strain history.

This notebook covers: Lagrangian vs Eulerian, `Particle`, `Field`, strain paths,
volumetric strain, and animated tracking.


In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from matplotlib.animation import FuncAnimation
from IPython.display import HTML
import geopyv_dev as gp

%matplotlib inline
plt.rcParams.update({
    "figure.dpi": 110,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "font.size": 11,
    "animation.embed_limit": 50,
})


In [ ]:
# Reload the sequence saved in notebook 04
# (If you haven't run notebook 04, generate + solve the sequence first.)
try:
    sequence = gp.load("data/sequence.pyv")
    print(f"Loaded: {sequence}")
except FileNotFoundError:
    print("Running sequence generation and solve (first time)...")
    import os, numpy as np
    os.makedirs("data/sequence", exist_ok=True)
    _sp = gp.Speckle(
        image_cfg={"image_dir": "data/sequence", "name": "seq", "image_size": (401, 401)},
        speckle_cfg={"speckle_size": 3.5, "speckle_number": 700},
        progression="deformation",
        deformation_cfg={"comp": [8.0,0.,0.,0.,0.,0.,0.,0.,0.,0.,0.,0.],
                          "mode": "SB", "option": "sin", "width": 100.0},
        noise_cfg=(0.2, 1.5),
        scale_cfg={"scale": "lin", "n": 5},
    )
    _sp.solve(seed=42)
    boundary = gp.PathRegion(nodes=np.array([[50.,50.],[50.,350.],[350.,350.],[350.,50.]]),
                              hard=False)
    sequence = gp.Sequence(image_dir="data/sequence", boundary=boundary,
                            target_nodes=60, size=(15., 70.), mesh_order=1)
    sequence.solve(local_mask=gp.Mask(mask_type="local", shape="circle", size=25),
                   seed_coord=[200., 200.], subset_order=1, tolerance=0.75,
                   options=gp.SequenceOptions(guide=True, sync=True))
    sequence.save("data/sequence.pyv")
    print(f"Done: {sequence}")


## Lagrangian vs Eulerian tracking

DIC gives you displacement vectors at **fixed node positions** for each frame.
There are two ways to integrate these into a strain path:

### Lagrangian (track=True — default)
The particle **moves with the material**.  Its coordinate updates after each increment
using the interpolated displacement at its current position.

```
frame 0:  x₀ = [200, 200]
frame 1:  x₁ = x₀ + u(x₀)   ←  displacement evaluated at x₀
frame 2:  x₂ = x₁ + u(x₁)   ←  displacement evaluated at new position x₁
```

### Eulerian (track=False)
The particle **stays fixed in space**.  The displacement at its original position
is evaluated for every frame.

```
frame 0:  x₀ = [200, 200]
frame 1:  u(x₀)
frame 2:  u(x₀)   ←  always the same reference point
```

The animation below illustrates the difference.


In [ ]:
# Conceptual Lagrangian vs Eulerian diagram
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 4.5), facecolor="white")

FRAMES = 4
DX = 6.0   # displacement per frame (px, simplified)

for ax, mode, color in [(ax1, "Lagrangian (track=True)", "seagreen"),
                         (ax2, "Eulerian  (track=False)", "royalblue")]:
    x = np.linspace(40, 360, 8)
    for xi in x:
        for frame in range(FRAMES + 1):
            alpha = 0.15 + 0.17 * frame
            ax.add_patch(patches.Circle((xi + frame * DX, 200 - frame * 1.5), 8,
                                         fc="grey", ec="none", alpha=alpha * 0.3))

    # Draw one tracked particle's path
    x0 = 200.
    if mode.startswith("L"):
        xs = [x0 + i * DX for i in range(FRAMES + 1)]
        ys = [200. - i * 1.5 for i in range(FRAMES + 1)]
    else:
        xs = [x0] * (FRAMES + 1)
        ys = [200.] * (FRAMES + 1)

    ax.plot(xs, ys, "o-", c=color, ms=10, lw=2, zorder=5)
    for i, (xi, yi) in enumerate(zip(xs, ys)):
        ax.text(xi + 3, yi - 10, f"$t_{i}$", fontsize=9, color=color)

    ax.set_xlim(0, 410); ax.set_ylim(150, 250)
    ax.set_aspect("equal"); ax.axis("off")
    ax.set_title(mode, fontsize=11, fontweight="bold", color=color)

plt.suptitle("Lagrangian vs Eulerian particle tracking", fontsize=12, fontweight="bold")
plt.tight_layout()
os.makedirs("../docs/assets", exist_ok=True)
fig.savefig("../docs/assets/nb5_lag_eul.png", dpi=120, bbox_inches="tight")
plt.show()


## The `Particle` class

```python
particle = gp.Particle(
    source     = sequence,          # solved Sequence or Mesh
    coordinate = [x, y],           # initial position [x, y]
    track      = True,             # True = Lagrangian, False = Eulerian
)
particle.solve(
    factor    = 0.0,               # volumetric correction factor
    true_incs = True,              # logarithmic strain increments
)
```

After solving, `particle.coordinates` has shape `(n_frames+1, 2)` — one row per frame
including the initial position.


In [ ]:
# Create a Lagrangian particle at the centre of the ROI
particle_lag = gp.Particle(source=sequence, coordinate=[200.0, 200.0], track=True)
particle_lag.solve(factor=0.0, true_incs=True)
print(particle_lag)
print(f"\n  coordinates shape : {particle_lag.coordinates.shape}")
print(f"  warps shape       : {particle_lag.warps.shape}")
print(f"  strains shape     : {particle_lag.strains.shape}")

# Eulerian version at the same location
particle_eul = gp.Particle(source=sequence, coordinate=[200.0, 200.0], track=False)
particle_eul.solve(factor=0.0, true_incs=True)
print(f"\nEulerian particle: {particle_eul}")


In [ ]:
# Plot coordinate trajectory and strain history
fig, axes = plt.subplots(1, 3, figsize=(14, 4))

# Trajectory (Lagrangian only)
coords = particle_lag.coordinates   # shape (n_frames+1, 2)
axes[0].plot(coords[:, 0], coords[:, 1], "o-", c="seagreen", ms=7, lw=1.5)
for i, (xi, yi) in enumerate(coords):
    axes[0].text(xi + 1, yi - 2, str(i), fontsize=9, color="seagreen")
axes[0].set_title("Lagrangian trajectory", fontsize=11, fontweight="bold")
axes[0].set_xlabel("x (px)"); axes[0].set_ylabel("y (px)")
axes[0].invert_yaxis()

# u-displacement over time: Lagrangian vs Eulerian
# warps shape: (n_frames+1, warp_len) — row 0 is initial; rows 1..N are post-solve
warp_frames = range(particle_lag.warps.shape[0])
u_lag = particle_lag.warps[:, 0]
u_eul = particle_eul.warps[:, 0]
axes[1].plot(warp_frames, u_lag, "o-", c="seagreen", label="Lagrangian", ms=6)
axes[1].plot(warp_frames, u_eul, "s--", c="royalblue", label="Eulerian", ms=6)
axes[1].set_title("u-displacement history", fontsize=11, fontweight="bold")
axes[1].set_xlabel("Frame"); axes[1].set_ylabel("u (px)"); axes[1].legend()

# Strain history: one row per solved increment
strains_lag = particle_lag.strains   # shape (n_frames, n_components)
n_str = strains_lag.shape[0]
strain_frames = list(range(1, n_str + 1))   # frames 1 … n_frames
axes[2].plot(strain_frames, strains_lag[:, 0], "o-", c="seagreen", label="exx Lag.")
axes[2].plot(strain_frames, strains_lag[:, 3], "s--", c="crimson",  label="eyy Lag.")
axes[2].set_title("Strain components", fontsize=11, fontweight="bold")
axes[2].set_xlabel("Frame"); axes[2].set_ylabel("Strain"); axes[2].legend()

plt.suptitle("Particle solution", fontsize=12, fontweight="bold")
plt.tight_layout()
fig.savefig("../docs/assets/nb5_particle.png", dpi=120, bbox_inches="tight")
plt.show()


## The `Field` class

`Field` places **many particles** across the mesh simultaneously and solves all of
them in one call.  By default it distributes particles at element centroids
(one per element) — use `gp.field_distribute_particles()` or supply explicit coordinates.

```python
field = gp.Field(sequence_solution=sequence, track=True, depth=1.0)
field.solve(factor=0.0, true_incs=True)
```

After solving, `field.particles` gives you a list of `ParticleSolution` objects,
one per particle.


In [ ]:
# Distribute particles at element centroids of pair-0 mesh
m0 = sequence.mesh_solution_at(0)
coords, volumes = gp.field_distribute_particles(
    nodes=m0.nodes, elements=m0.elements, depth=1.0
)
print(f"Distributed {len(volumes)} particles across {m0.elements.shape[0]} elements")

field = gp.Field(sequence_solution=sequence, track=True, depth=1.0,
                  coordinates=coords, volumes=volumes)
field.solve(factor=0.0, true_incs=True)
print(field)
print(f"  n_particles : {field.n_particles}")
print(f"  inc_no      : {field.inc_no}")


In [ ]:
# Final-frame volumetric strain map
particles = field.particles

# Each particle has accumulated strain; take the last increment
final_vol_strains = np.array([p.vol_strains[-1] for p in particles])
init_coords = field.coordinates   # (N, 2) initial positions

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))

sc1 = ax1.scatter(init_coords[:, 0], init_coords[:, 1],
                   c=final_vol_strains, cmap="RdBu_r", s=40, linewidths=0)
plt.colorbar(sc1, ax=ax1, fraction=0.046, label="εᵥₒₗ")
ax1.set_title(f"Volumetric strain — frame {field.inc_no}", fontsize=11, fontweight="bold")
ax1.set_aspect("equal"); ax1.set_xlim(0, 401); ax1.set_ylim(0, 401); ax1.invert_yaxis()
ax1.axis("off")

# u-displacement at final frame per particle
final_u = np.array([p.warps[-1, 0] for p in particles])
sc2 = ax2.scatter(init_coords[:, 0], init_coords[:, 1],
                   c=final_u, cmap="RdBu_r", s=40, linewidths=0)
plt.colorbar(sc2, ax=ax2, fraction=0.046, label="u (px)")
ax2.set_title("u-displacement — final frame", fontsize=11, fontweight="bold")
ax2.set_aspect("equal"); ax2.set_xlim(0, 401); ax2.set_ylim(0, 401); ax2.invert_yaxis()
ax2.axis("off")

plt.suptitle("Field solution — per-particle quantities", fontsize=12, fontweight="bold")
plt.tight_layout()
fig.savefig("../docs/assets/nb5_field.png", dpi=120, bbox_inches="tight")
plt.show()


In [ ]:
# Animated: volumetric strain evolving over all frames
all_vol_strains = np.array([[p.vol_strains[t] for p in particles]
                             for t in range(field.inc_no)])   # shape (n_frames, n_particles)
vmin_v, vmax_v = all_vol_strains.min(), all_vol_strains.max()

fig, ax = plt.subplots(figsize=(6, 6))
sc = ax.scatter(init_coords[:, 0], init_coords[:, 1],
                c=all_vol_strains[0], cmap="RdBu_r", s=40, linewidths=0,
                vmin=vmin_v, vmax=vmax_v)
cb = plt.colorbar(sc, ax=ax, fraction=0.046, label="εᵥₒₗ")
ttl = ax.set_title("Frame 1: volumetric strain", fontsize=11, fontweight="bold")
ax.set_aspect("equal"); ax.set_xlim(0, 401); ax.set_ylim(0, 401); ax.invert_yaxis()
ax.axis("off")

def update_field(frame):
    sc.set_array(all_vol_strains[frame])
    ttl.set_text(f"Increment {frame + 1}: volumetric strain")
    return sc, ttl

anim_f = FuncAnimation(fig, update_field, frames=field.inc_no, interval=600, blit=True)
plt.tight_layout()
anim_f.save("../docs/assets/nb5_vol_strain.gif", writer="pillow", fps=2)
HTML(anim_f.to_jshtml())


In [ ]:
# Save and reload
field.save("data/field.pyv")
field_loaded = gp.load("data/field.pyv")
print(f"Loaded: {field_loaded}")
print(f"  n_particles : {field_loaded.n_particles}")
print(f"  inc_no      : {field_loaded.inc_no}")


## Summary: the full workflow

```
gp.Speckle(...)          → generate synthetic images (or use real ones)
gp.Image(filepath=...)   → load images

gp.Mask(mask_type="local", shape="circle", size=25)    → subset template

gp.Sequence(image_dir, boundary, target_nodes, ...)
    .solve(local_mask, seed_coord, ...)               → correlation over time

gp.Field(sequence, track=True, coordinates=coords, volumes=vols)
    .solve()                                          → strain paths for all particles

gp.save("result.pyv", field)   /   gp.load("result.pyv")
```

From here you can explore:
- **Calibration** (`gp.CalibrationParams`) to convert pixel displacements to physical units.
- **Validation** (`gp.Validation`) against known synthetic deformations.
- **Visualisation** via `geopyv_dev.plots` — `contour_field`, `history_field`, `trace_field`.
